RICONOSCIMENTO FACCIALE: APPROCCIO TRADIZIONALE vs DEEP LEARNING

Passiamo dal vedere all'intendere

Face detection:
C'è un volto nell'immagine? dove si trova?
Face recognition:
Di chi è questo volta?

Storicamente, il riconoscimento facciale, si basava sulla misurazione di rapporti geometrici tra punti focali come occhi, naso e bocca.
Si cercavno dei rapporti tra i vari elementi, il problema, prova a sorridere o girare la testa, quei rapporti cambiano.
Oggi non cerchiamo più di spiegare alla macchina cosìè un naso (con la geometrica), lasciamo che sia lei ad individuare una rappresentazione numerica chiamata embedding

Il Deep Learning ha sostituito queste misure rigide con vettori numerici chiamati embeddings, generati da reti neurali convoluzionali



1. Approccio tradizonale (prima del Deep Learning)
Prima del Deep Learning il riconoscimeto facciale era basato soprattutto su caratteristiche progettate matematicamente.
Volto -> grayscale -> normalizzazione -> estrazione caratteristiche -> classificatore
Gli algoritmi più importanti sono:
- Haar Cascade
- Eigenfaces
- Fisherfaces
- LBPH
Il modello recupera le coordinate del rettangolo che contiene il volto



* Haar Cascade * 
Haar Cascade viene normalmente utilizzato per rilevare il volto, non per riconoscere l'identità. Il modello non dice qui c'è PincoPallino, ma dice qui c'è un volto.

* Eigenface *
Eigenfaces è un metodo storico basato sulla PCA (Principal Component Analysis).
Immagina di avere un'immagine facciale da 100x100 pixel, hai 10.000 valori, troppi.
Con la PCA estrai le n caratteristiche principali, che rappresentano le variazioni più importanti di un volto
10.000 pixel -> 100 caratteristiche
Faccia originale -> PCA -> vettore di n caratteristiche
Ogni volto diventa quindi un vettore numerico che può essere confrontato con altri vettori facciali.
Il problema è che Eigenface è molto sensibile a illuminazione, posizoine, inclinazione volto, espressione, occhiali, sfondo, ecc.
Quindi funziona decentemente solo in ambienti e posizioni controllate.

* Fisherfaces *
Fisherfaces cerca di migliorare Eigenfaces
Mentre PCA cerca principalmente le direzini con maggiore varianza, Fisherface utilizza l'informazione sulle classi.
Fisherface cerca una rappresentazione che avvicini le immgini della stessa persona, e contemporaneamente allontani persone diverse. Formalmente è basato sulla Discriminant Analysis LDA
PCA -> massimizza la varianza
LDA -> massimizza la separazione tra classi

* LBPH *
Local Binary Patterns Histograms
Algoritmo classico e reso popolare da OpenCV.
Prende un pixel centrale e confronta i pixel vicini
Per esempio
10 20 15
30 25 40
15 35 20
centro 25 e confronta con i pixel vicini
pixel >= 25 ->1
pixel <25 ->0
ottengo
0 0 0
1   1
0 1 0
l'immagine viene suddivisa in regioni e viene cotruito un istogramma di questi pattern
Vantaggi: semplice, veloce, poco costoso, abbastanza efficiente con dataset piccoli, utilizzabile anche senza GPU
Svantaggi: soffre di poso diverse, illuminazione difficile, età, espressioni, qualità immagini.

ARRIVA IL DEEP LEARNING

Cambia tutto.
Nei sistemi tradizionali se tu, o il progettista dell'algoritmo, a stabili quali caratteristiche estrarre. Esempio PCA, texture, gradienti, LBP
Nel Deep Learning la rete impara automaticamente quali caratteristiche sono utili.
E' lo stesso principio visto nella CNN
Immagine -> Conv2D -> bordi -> forme -> occhi/naso/bocca -> caratteristiche più astratte -> identità

Ma i sistemi moderno fanno anche qualcosa di più

Non producono il risultato Barbara (riconoscimento facciale)
ma producono un EMBEDDING


*** FACE EMBEDDING ***

Un volto viene trasformato in un vettore numerico
Esempio
Barbara.jpg
[0.123,0.082,...,0.531]
Potrebbe essere un vettore di 128, 512 o più valori a seconda del modello.
Questo vettore rappresenta, matematicamente, il volto
E' esattamente lo stesso concetto degli embedding visti nel NLP

Il principio è identico:
Elementi semanticamente simili vengono rappresentati da vettori vicini
Fotografie della stessa persona dovrebbero produrre embedding vicini.
Esempio
Barbara.jpg= [0.123,0.082,...,0.531]
Erika.jpg =[0.8,0.1,0.5,...,0.8]
Calcoli la distanza tra i due embedding
Se la distina è minore di una certa soglia allora consideri i due embedding (i due volti) appartenenti alla stessa persona.

Quindi, con l'arrivo del Deep Learning, per capire se due foto appartengono alla stessa persona, non cerchiamo più simiglianza visiva, ma calcoliamo la distanza tra gli embedding 

Ma come fa la rete ad imparare dove posizionare questi punti in modo così preciso?

Il segreto è il Deep Metric Learning

Il Deep Metric Learning cerca di organizzare lo spezio degli embedding in modo che oggetti simili siano vicini, ed oggetti diversi siano lontani.
Questa è la metrica che viene 'imparata'. FaceNet per esempio, è stato progettato proprio per mappare direttamente i volto in uno spazio euclideo nella quale la distanza rappresenta la similarità facciale.
Non vogliamo che la rete impari la classe 'barbara' e la classe 'erika'
vogliamo che la reti imparti che stessa identità -> distanza piccola  identità diverse -> distanza grande.

Questa è la differenza fondamentale rispetto alla classificazione

Con il classificatore se addestro una reta ad imparare la classe  Barbara e la classe Erika, se domani ho fato di una terza persona devo riaddestrare il modello per imparare la classe Barbara, la classe Erika, la classe AltraPersona.

Con il metric learning invece cambia tutto
La rete, ha già prodotto gli embedding Barbara e gli embedding Erika, ora deve produrre gli embedding AltraPersona. Salvi il nuovo embedding nel database, il tutto senza dover creare una nuova classe o riaddestare il modello.

il vero problema ora, quindi, diventa: come insegno alla rete quali vettori devono essere vicini e quali lontano?
Qui entrano in gioroc le loss specifiche del Deep Metric Learning

Immagia una galleria d'arte dove i ritratti vengono continuamente spostati per raggruppare quelli che appartengono allo stesso sogggetto.
Utiliziamo un trucco chiamato TRIPLET LOSS, la funzione di perdita che addestra la rete avvicinando l'anchor all'esempio positivo e allontandnando da quello negativo.

QUI ENTRA IN GIOCO FACENET
Uno dei lavori fondamentali del riconoscimento facciale moderno è FaceNet
L'idea è: 
immagine del volto -> rete neurale -> embedding
FaceNet è diventato famoso anche per l'utilizzo della TRIPLET LOSS

Consideriamo 3 immagini:
Anchor - Barbara
Positive - Barbara
Negative - Erika
La  rete deve imparare 
distanza(barbara,barabara) - piccola
distanza(barbara,erika) - grande
barbara ---- barbara      erika
Lo spazizo degli embedding viene organizzato in modo che le immagini della stessa persona si raggruppino
Quesot è estremamente potente.

* ArcFace * 
Una famiglia successiva di metodi, molto importante, utilizza tecniche come ArcFace.
ArcFace cerca di creare embedding ancora più disciminanti.
Vuole ottenere embedding molto vicino se della stessa persona, molto lontani se di altre persone, creando dei custer, riducendo la sovrapposizione tra persone differenti.

Esiste però un modo ancora più sofisticato per confrontare questi vettore, specialmente quando le dimensioni aumentano

Ottimizzazione delle Ditanza

In uno spazio con tanti vettori, la distanza tra vettori può ingannare, ecco perchè spesso si preferisce usare il coseno di similitudine per confrontare vettori normalizzati in sistemi ad alta scala.
Questo metodo MISULA L'ANGOLO TRA I VETTORI, ignorando la magnitudo e focalizzandosi puramente sull'orientamento delle feature

Invece di misurare quanto sono distanti due punti, misuriamo l'angolo tra due vettori che partono dall'origine.
E' come se non ci importasse quanto forte è il segnale, ma in che direzione punta.
Questo rendo il sistema robusto ai cambiamenti globali di luminosità per esempio, perchè l'orientamento dei tratti rimane lo stesso.

Ora che abbiao campito come rappresentare un volto, vediamo come arrivarci partendo da un'immagine grazza

La Pipeline di Riconoscimento

La pipelne di riconoscimento facciale, è la sequenza completa che trasforma una foto o un frame video in una decisione del tipo 'questa persona è Barbara' oppure 'persono non riconosciuta'
Il riconoscimento non è un singolo algoritmo, è una catena di passaggi, e se uno funziona male, tutto il sistema peggiora.
La pipeline moderna tipica è:
acquisizione immagine -> face detection -> face alignment -> preprocessing -> feature extraction -> embedding -> matching -> threshold -> Identica/unknow

* Acquisizione immagine
La sorgente può essere: foto, webcam, telecamera IP, video, smarphong
Esempio 
import cv2
frame=cv2.imread("persona.jpg")
oppure (webcam) 
cap=cv2.VideoCapure(0)
ret,frame=cap.read()
Una telecamera pessima, un volto di 30 pixel o una luce fortemente contorluce, non vengono magicamente risolti dal Deep Learning
La qualità conta moltissimo

* Face Detectiont
Prima devi trovare il volto
Input: immagine completa
output: bounding box del volto
Il detector non dice chi è la persona, dice solo qui c'è un volto

* Face Alignment
Supponiamo di avere due immagini della stessa persona, la prima immagine a volto dritto, la seconda volto inclinato
Se confrontassi direttamente le due immagini, le differenze sarebbero grandi.
L'allineamento cerca punti di riferimento, chiamati facial landmarks
Indicativamente: occhi, naso, bocca
Il volto viene quindi ruotato, ridimensionato, centrato, in una configurazione standard.
Questo riduce le variazioni inutili
E' un concetto simile alla noramalizzazione dei dati nel Machine Learning

* Crop
Una volta trovato il volto, una volta 'normalizzato' elimino gran parte dell'immagine inutile.
esempio, ufficio, scrivania, parate, per ottenere solo il volto
Questo è importante perchè il modello deve riconoscere la persona non il suo ufficio.

* Preprocessing
Il volto viene adattato al formato richiesto dal modello.
Esempio 112x112 oppure 160x160 oppuer 224x224, dipende dal modello scelto
Poi normalizzi i pixel da 0-255 a 0-1 oppure a -1 1
Non esiste una normalizzazione universale, devi usare quella richiesta dal modello

* Feature Extraction
Qui entra in gico il Deep Learning, la rete riceve un volto normalizzato, e analizza caratteristiche progressivamente più complesse 
in una CNN
dobbiamo prima torvare i bordi, estrerre le caratteristiche, ed infine confrontarle

pixel -> bordi -> texture -> forme -> struttura del volto -> caratteristiche biometriche

A differenza degli algoritmi tradizionali, queste caratteristiche vengono apprese durante il training
Non dici: guarda la distanza fra gli occhi
La rete decide autonomamente quali combinazioni di caratteristiche siano simili

* Creazione dell'embedding
Il risultato non è Barbara ma un vettore che potrebbe avere 128 o 512 dimensioni, dipende dal modello
Idealmente due embedding di due foto della stessa persona dovrebbero essere simili, mentre di persone diverse dovrebbero essere distanti

* Enrollment

Prima di riconoscere la persona, devi costruire il databse delle perosne conosciute
per esempio raccogli n foto di Barbara, calcolo tutti i n embed
puoi salvare, per esempio, l'embedding medio

* Matching
Arriva una nuova immagine
Calcolo l'embedding e lo confronti con i vettori già registrati calcolando la distanza euclidea

Ma non devi solo dire, questa nuova immagine corrisponde a Barbara (perchè le differenze sono minori rispetto ad altre immagini), perchè se arrivasse l'immagine di una nuova persona, questa non sarebbe nel databse, ma ugualmente avrà un embedding più vicino rispetto agli altri embedding
Devi pertanto scegliere una soglia utilizzata per confrontarla con la differenza tra due embedding

Con questa soglia determini anche la precisione del modello
In certi contesti è meglio avere un False Rejection che un False Acceptance
Meglio un 'persona non riconosciuta' che 'questa è Barbara'
Quindi la soglia dipende dal contesto

Dallo Scatto alla Firma Digitale
Stadi della pipeline moderna
Tutto inizia con la FACE DETECTION: localizzazione del volto nell'immagine tramite algoritmo come MTCNN o modelli basati su Transformer
Poi ruotiamo e scaliamo il volto per centrare occhi, bocca, naso, rendendo la rappresentazioe invarinte alla posa (allignment).
Solo a questo punto l'immagine pulita entra nella CNN per la rapresentation, dove viene trasformato in un vettore.
Confrontiamo il vettore così creato all'interno del nostro database, tramite classificatori e calcolo di distanza.

Fermiamoci sull'allineamento
E' davvero così fondamentale?

Importanza dell'Allineamento
L'allinemaento è fondamentale, pensa ad un fotografo che vi chiede di spostarvi per una foto-tessera, l'allineamento fa lo stesso digitalmente.

La Landmarks Deteciont è l'identificazione di punti chiave come angoli degli occhi, punta del naso, necessari per calcolare la trasformazione geometrica necessaria.
La Trasformazione Affine è l'operazione matematica che ruota e scala l'immagine afficnè le caratteristche facciali siano sempre nella stessa posizone relativa. Con la trasformazione affine modifichiamo leggermente l'immagine per far si che il volto appaio sempre centrale ed allineato per la rete
Il volto viene ritagliato e portato a una risoluzione fissa, solitamente 112x112 o 224x224 pixel, prima dell'inferenza (normalizzazione e Crop)

Una volta allineato e centrato il volto ha bisogno di un'ultima rifinitura.

Invarianza e Robustessa
Normalizzazione dei vettori
Per rendere il confronto indipendente dalla luminosità globale, i vettori di embedding vengono spesso normalizzati a norma unitaria.
In questo modo, tutti i punti della spazio latente ridiedono sulla superficie di una ipersfsera n-dimensionale.
Prendiamo il nostro vettore e lo scaliamo finchè la sua lunghezza totale sia esattamente 1, immagina ora che tutti i volti del mondo siano ora mappati sulla superficio di una gigantesca sfera invisibile.

In questo modo, confrontare due persone diventa un semplice calcolo geometrico sulla stessa sfera.
Rendendo la ricerca nel databse fulminea

Tutto questo funziona bene in foto perfette

Le Sfide In The Wild
Affrontare la variabilità del mondo reale.
Nel mondo reale le persone non collaborano.
Indossano occhiali da sole, mascherine, zone ombra o girano la testa.
In un sistema che funziona solo sotto le luci di uno studio fotografico è inutile per la sicurezza di un aeroporto o per il riconoscimento facciale dello smartphone.
Dobbiamo progettare reti che sappiano immagiina parte mancannt volto

Analiziamo i 3 nemici principali del riconoscimento di  un volto
- Occlusione: presenza di accessori come occhiali, cappelli, o mascchene che nascondono porzioni critiche del volto
- Illuminazione: ombre scure o sovraesposizione possono nascondere le texture cutanee essenziali per la discriminazione
- Posa: rotazioni estreme del capo che impediscono la visualizzazione simmetrica dei tratti facciali.
- Espressioni: deformazioni elastiche del volto che alterano la posizione dei landmarks rispetto alla posa neutra.

Per combattere que4sti problemi, addestriamo le reti con UNA FUNZIONE DI PERDITA ed un margine di sicurezza.

Ma come prepariamo i nostri modelli a queste difficoltà durante l'addestramento?

Strategia di Mitigazione
- Data Augmentation: addestramento delle reti con immagini manipolate artificialmente per simulare ombre, rotazioni e rumore, sfumature
- Modelli multi-view: sistemi che utilizzano più angolazioni contemporaneamente per ricostruire una rappresentazione 3D del volto
- Sintesi dei volti: utilizzo di GAN per generare pose neutre partendo da immagini di profilo, facilitando il compito della rete di riconoscimento.

Una delle sfide più recenti è stata quella della mascherine chirurgiche
La pandemia ha costretto i ricercatori a riscrivere le reti.

Le reti moderne vengono istruite a dare maggiore peso statistico all'area degli occhi quando la parte inferiore del volto è occlusa
Questo approccio garantisce la continuità del servizio anche in scenari sanitari o di sicurezza pubblica complessi.
E' emerso che gli occhi contengono informazini identitarie sufficienti per la maggior parte delle applicazioni

Una buona rete neurale è flessibile, sa dove guardare 








